# P1 三つの呼び先 ― 模擬・再生・本物

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

同じコードを、模擬（mock）・講師の記録の再生（replay）・本物の API（anthropic）で動かし分けます。題材は P3 で詳しく見る、問い合わせ分類（ソラハ電機・6 ラベル）です。

- 上から順に実行します。0 のセルは準備です（Colab でも手元の Jupyter でも同じ）。
- API キーが無ければ講師の記録の再生（replay）で、本物の実行と同じ数字が出ます（費用 0）。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る（Colab のときだけ取得とインストールをする）
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/llm-eval-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/llm-eval-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根（common/ がある所）を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/llm.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. AI の呼び先を決める（キーは表示しない）
from common import nb
from common.nb import choose_provider, safe, show, show_file, lab
PROVIDER = choose_provider(IN_COLAB)

## 1. 模擬（mock）で 6 件を分類する
キーも記録も要りません。決まった筋書きで答えるので、数字は性能の証拠になりません。

In [ ]:
from common import kit, llm as L
rc = lab("run_classify", "lab03_classify_extract")
golden = kit.load_jsonl("golden_inquiries.jsonl")
args = kit.parser("demo").parse_args(["--provider", "mock"])
with L.mock_handler(rc.mock_for(golden)):
    runs = [rc.classify_one(r, args) for r in golden[:6]]
for x in runs:
    print(x["id"], f'{x["label"]:<8}', "→", f'{x["pred"]:<8}', "○" if x["correct"] else "×")

## 2. 講師の記録を再生（replay）して、同じ 6 件を分類する
本物の Opus 5.5 の応答を `.cache/llm/` から返します。費用 0、待ち時間と費用は元の実行の値です。

In [ ]:
args = kit.parser("demo").parse_args(["--provider", "replay" if PROVIDER == "mock" else PROVIDER,
                                       "--model", "claude-opus-5-5"])
with safe():
    runs = [rc.classify_one(r, args) for r in golden[:6]]
    for x in runs:
        print(x["id"], f'{x["label"]:<8}', "→", f'{x["pred"]:<8}', "○" if x["correct"] else "×",
              f'元の費用 {x["cost_usd"]:.4f} USD / {x["seconds"]:.1f} 秒 / 記録から {bool(x["cached"])}')

## 3. 120 件を 2 つのモデルで再生し、RESULTS.md の数字と比べる
キー無しでも、講師が API で回したときと同じ正答数と費用が出ます。

In [ ]:
with safe():
    for model in ("claude-sonnet-5-5", "claude-opus-5-5"):
        a = kit.parser("demo").parse_args(["--provider", "replay" if PROVIDER == "mock" else PROVIDER, "--model", model])
        runs = [rc.classify_one(r, a) for r in golden]
        ok = sum(x["correct"] for x in runs)
        print(f"{model:<18} 正答 {ok}/{len(runs)}  費用 {sum(x['cost_usd'] for x in runs):.4f} USD")
print("RESULTS.md: Sonnet 120/120・0.37 USD / Opus 120/120・0.83 USD")